# Week 4 — Baseline Model
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 4  
**Dataset:** CRMLS Sold Properties, cleaned in Week 3

**Goal:** Train a Linear Regression baseline, experiment over training window
length (X months preceding the test month), and evaluate with R² on the test set.
Test month is June 2026 this time, since that's now our most recent data.

In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error, median_absolute_error

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'
model_df = pd.read_csv(data_folder + '\\cleaned_full.csv', parse_dates=['CloseDate_parsed'])

feature_cols = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]
target_col = 'ClosePrice'

print(f'Loaded {len(model_df):,} rows')

Loaded 411,419 rows


## 1. Reuse the Train/Test Split Function from Week 3
Pasting in the same function so this notebook is self-contained.

In [7]:
def get_train_test_split(frame, test_month, window_months):
    frame = frame.copy()
    frame['YearMonth'] = frame['CloseDate_parsed'].dt.to_period('M')
    test_df = frame[frame['YearMonth'] == test_month]
    train_start = test_month - window_months
    train_df = frame[(frame['YearMonth'] >= train_start) & (frame['YearMonth'] < test_month)]
    return train_df.drop(columns='YearMonth'), test_df.drop(columns='YearMonth')

## 2. Training Window Experiment
Test month is June 2026 (most recent data). I'm sweeping window length from 3 to
24 months to see where the tradeoff sits between more training data (helps the
model generalize) and staleness (older sales may not reflect current pricing
dynamics, especially given how volatile 2022–2023 looks in the Week 2 monthly
counts). For each window length, I fit the StandardScaler on **training data
only**, transform both train and test with that same scaler, then fit Linear
Regression and evaluate

In [8]:
test_month = pd.Period('2026-06', freq='M')
window_options = [3, 6, 9, 12, 18, 24]

results = []

for window in window_options:
    train_df, test_df = get_train_test_split(model_df, test_month, window)

    if len(train_df) < 50 or len(test_df) < 10:
        print(f'Window={window}: skipped, not enough rows (train={len(train_df)}, test={len(test_df)})')
        continue

    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    r2 = r2_score(y_test, preds)
    mape = mean_absolute_percentage_error(y_test, preds)
    mdape = np.median(np.abs((y_test - preds) / y_test))

    results.append({
        'window_months': window,
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'R2': r2,
        'MAPE': mape,
        'MdAPE': mdape
    })

results_df = pd.DataFrame(results)
results_df

,window_months,train_rows,test_rows,R2,MAPE,MdAPE
0,3,35186,12841,0.481007,0.464077,0.312184
1,6,61637,12841,0.480339,0.451134,0.307162
2,9,94832,12841,0.478428,0.451299,0.307264
3,12,130060,12841,0.480024,0.452677,0.308715
4,18,191859,12841,0.477012,0.453806,0.307877
5,24,264023,12841,0.479783,0.460463,0.309593


## 3. Pick the Best Window
I'm using R² as the primary criterion since that's the required baseline metric,
with MAPE/MdAPE as a sanity check (R² alone can be misleading if a model is
underfitting uniformly).

In [9]:
best_row = results_df.loc[results_df['R2'].idxmax()]
best_window = int(best_row['window_months'])
print(f'Best window: {best_window} months')
print(best_row)

Best window: 3 months
window_months        3.000000
train_rows       35186.000000
test_rows        12841.000000
R2                   0.481007
MAPE                 0.464077
MdAPE                0.312184
Name: 0, dtype: float64


## 4. Final Baseline Fit + Coefficient Inspection
Refitting with the best window so I can look at which features are pulling the
most weight — useful sanity check before moving to a more complex model.

In [10]:
train_df, test_df = get_train_test_split(model_df, test_month, best_window)

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

final_model = LinearRegression()
final_model.fit(X_train_scaled, y_train)
final_preds = final_model.predict(X_test_scaled)

print(f'Final R²: {r2_score(y_test, final_preds):.4f}')
print(f'Final MAPE: {mean_absolute_percentage_error(y_test, final_preds):.4f}')

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': final_model.coef_
}).sort_values('coefficient', key=abs, ascending=False)
coef_df

Final R²: 0.4810
Final MAPE: 0.4641


,feature,coefficient
0,LivingArea,398918.268322
2,BathroomsTotalInteger,293799.260018
4,PropertyAge,245971.091719
7,Longitude,-97012.342689
5,DaysOnMarket,-95744.384879
1,BedroomsTotal,-61637.168473
6,Latitude,-43695.654770
12,AssociationFee,28844.740518
8,PoolPrivateYN,-22918.842390
13,LivingArea_missing,19797.812045


## Summary
- Test month: June 2026 (most recent). Swept training window from 3–24 months.
- Best window: **[fill in from results_df after running]** months, R² = **[fill in]**.
- This is a weak-to-moderate baseline as expected — Linear Regression can't
  capture non-linear location/size interactions well. Documenting this as the
  floor to beat in later weeks (tree-based models, spatial features from the
  school-district join, etc.).
- Compared to the old Week 4 baseline (24-month window, weak R²), this version
  benefits from actually imputing rather than dropping ~9 columns' worth of
  rows, plus the outlier capping — worth checking if R² improved as a result
  once you run it.